In [1]:
import pandas as pd
import os
import csv
import json # Import json for loading dataset.json

# --- Configuration ---
# --- Pandas Display Options (Add these lines) ---
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 5000)      # Set a wider display width to prevent truncation
pd.set_option('display.max_rows', None)   # Display all rows (if matching_matches has many rows)
# pd.set_option('display.colheader_justify', 'left') # Optional: Adjust column header alignment

# Set the directory for your main Premier League data
DATA_DIR = './LiveSum_++/english-premier-league/'
# Set the directory for your individual match stat files (e.g., sample_1_table.csv)
SAMPLE_DATA_DIR = './LiveSum_++/training_data/'
# Set the directory for your dataset.json file
DATASET_JSON_PATH = './LiveSum_++/dataset.json' # Define path for dataset.json

# List of CSV files to process for the main historical data
CSV_FILES = [f'{year}-{str(int(year) + 1)[-2:]}.csv'
             for year in range(2013, 2024)] # Generates '2013-14.csv', '2014-15.csv', etc. up to '2021-22.csv'

# --- Dynamic TARGET_STATS Population Function ---
def populate_target_stats_from_csv(file_number, data_directory):
    """
    Populates the TARGET_STATS dictionary dynamically from a single CSV file
    with 'Away Team' and 'Home Team' data on separate rows.

    Args:
        file_number (int): The number of the CSV file to read (e.g., 1 for sample_1_table.csv).
        data_directory (str): The directory where the CSV files are located (e.g., SAMPLE_DATA_DIR).

    Returns:
        tuple: A tuple containing the dynamically populated TARGET_STATS dictionary and
               a list of column names extracted from the sample CSV's header (excluding 'Team').
               Returns (None, None) if the file/data is not found or an error occurs.
    """
    file_name = f"sample_{file_number}_table.csv"
    file_path = os.path.join(data_directory, file_name)

    dynamic_target_stats = {}
    sample_columns = []

    try:
        with open(file_path, mode='r') as csvfile:
            reader = csv.reader(csvfile)

            # Read the header row and strip whitespace
            header = [h.strip() for h in next(reader)]

            # Find the indices of the columns we need
            column_indices = {}
            for i, col_name in enumerate(header):
                column_indices[col_name] = i
                if col_name != 'Team': # Collect column names from sample, excluding 'Team'
                    sample_columns.append(col_name)

            if 'Team' not in column_indices:
                return None, None

            away_data = None
            home_data = None

            # Read the next two rows (Away Team and Home Team)
            try:
                row1 = [d.strip() for d in next(reader)]
                row2 = [d.strip() for d in next(reader)]
            except StopIteration:
                return None, None

            # Determine which row is Away and which is Home
            team_col_idx = column_indices['Team']
            if row1[team_col_idx].lower() == 'away team' and row2[team_col_idx].lower() == 'home team':
                away_data = row1
                home_data = row2
            elif row1[team_col_idx].lower() == 'home team' and row2[team_col_idx].lower() == 'away team':
                away_data = row2
                home_data = row1
            else:
                return None, None

            # Helper to safely get integer value
            def get_stat_value(data_row, stat_name_in_csv):
                if stat_name_in_csv in column_indices:
                    try:
                        return int(data_row[column_indices[stat_name_in_csv]])
                    except ValueError:
                        return None
                return None

            # Populate dynamic_target_stats for Away Team
            goals_away = get_stat_value(away_data, 'Goals')
            if goals_away is not None: dynamic_target_stats['FTAG'] = {'operator': '==', 'value': goals_away}
            shots_away = get_stat_value(away_data, 'Shots')
            if shots_away is not None: dynamic_target_stats['AS'] = {'operator': '==', 'value': shots_away}
            fouls_away = get_stat_value(away_data, 'Fouls')
            if fouls_away is not None: dynamic_target_stats['AF'] = {'operator': '==', 'value': fouls_away}
            yellow_away = get_stat_value(away_data, 'Yellow Cards')
            if yellow_away is not None: dynamic_target_stats['AY'] = {'operator': '==', 'value': yellow_away}
            red_away = get_stat_value(away_data, 'Red Cards')
            if red_away is not None: dynamic_target_stats['AR'] = {'operator': '==', 'value': red_away}
            corners_away = get_stat_value(away_data, 'Corner Kicks')
            if corners_away is not None: dynamic_target_stats['AC'] = {'operator': '==', 'value': corners_away}

            # Populate dynamic_target_stats for Home Team
            goals_home = get_stat_value(home_data, 'Goals')
            if goals_home is not None: dynamic_target_stats['FTHG'] = {'operator': '==', 'value': goals_home}
            shots_home = get_stat_value(home_data, 'Shots')
            if shots_home is not None: dynamic_target_stats['HS'] = {'operator': '==', 'value': shots_home}
            fouls_home = get_stat_value(home_data, 'Fouls')
            if fouls_home is not None: dynamic_target_stats['HF'] = {'operator': '==', 'value': fouls_home}
            yellow_home = get_stat_value(home_data, 'Yellow Cards')
            if yellow_home is not None: dynamic_target_stats['HY'] = {'operator': '==', 'value': yellow_home}
            red_home = get_stat_value(home_data, 'Red Cards')
            if red_home is not None: dynamic_target_stats['HR'] = {'operator': '==', 'value': red_home}
            corners_home = get_stat_value(home_data, 'Corner Kicks')
            if corners_home is not None: dynamic_target_stats['HC'] = {'operator': '==', 'value': corners_home}

            return dynamic_target_stats, sample_columns

    except FileNotFoundError:
        print(f"Error: Sample file not found at {file_path}")
        return None, None
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, None

## Core Data Loading and Filtering Functions
def load_and_combine_data(data_directory, csv_files, columns_to_keep=None):
    """
    Loads multiple CSV files from a directory into a single Pandas DataFrame.
    Assumes CSVs contain Premier League match data.
    Optionally drops columns not in `columns_to_keep`.
    """
    all_data = []
    for filename in csv_files:
        filepath = os.path.join(data_directory, filename)
        if os.path.exists(filepath):
            try:
                df = pd.read_csv(filepath, encoding='latin1')
                df['Season'] = filename.split('.')[0] # Add a 'Season' column

                if columns_to_keep:
                    cols_to_drop = [col for col in df.columns if col not in columns_to_keep and col != 'Season']
                    if cols_to_drop:
                        df = df.drop(columns=cols_to_drop)

                all_data.append(df)
            except Exception as e:
                print(f"Error loading {filename}: {e}")
        else:
            print(f"File not found: {filename}")

    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        return combined_df
    else:
        print("No data loaded. Please check your DATA_DIR and CSV_FILES list.")
        return pd.DataFrame()

def find_matches_by_stats(dataframe, target_stats):
    """
    Filters a DataFrame to find matches that meet the specified statistical criteria.
    """
    if dataframe.empty:
        return pd.DataFrame()

    condition = pd.Series(True, index=dataframe.index)

    for col, criteria in target_stats.items():
        if col in dataframe.columns:
            operator = criteria['operator']
            value = criteria['value']

            # Ensure the column's data type is compatible for comparison
            # Attempt to convert to numeric if not already, ignoring errors
            if pd.api.types.is_numeric_dtype(dataframe[col]):
                pass # Already numeric, no conversion needed
            else:
                try:
                    dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce')
                except TypeError:
                    print(f"Warning: Cannot convert column '{col}' to numeric for comparison.")
                    continue # Skip this column if conversion fails

            # Apply condition only if value is not None (from get_stat_value) and column is not NaN
            if value is not None:
                if operator == '>':
                    condition &= (dataframe[col] > value)
                elif operator == '<':
                    condition &= (dataframe[col] < value)
                elif operator == '==':
                    condition &= (dataframe[col] == value)
                elif operator == '>=':
                    condition &= (dataframe[col] >= value)
                elif operator == '<=':
                    condition &= (dataframe[col] <= value)
                elif operator == '!=':
                    condition &= (dataframe[col] != value)
                else:
                    pass
        else:
            pass

    return dataframe[condition]

# --- Main Execution Logic for Automation ---
if __name__ == "__main__":
    print(f"Using main data directory: {DATA_DIR}")
    print(f"Using sample data directory: {SAMPLE_DATA_DIR}")
    print(f"CSV files to process (main data): {CSV_FILES}")

    # Create a set of columns expected in the main dataframe, derived from the sample's relevant columns
    # and common match identifiers.
    main_df_relevant_columns = set([
        'HomeTeam', 'AwayTeam', 'FTR', 'Div', 'Date', 'Referee', # Common match identifiers
        'FTHG', 'FTAG', 'HS', 'AS', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'HC', 'AC', 'HTHG', 'HTAG', 'HTR',
        # Include all relevant stat columns for display
        'Shots', 'ShotsOnTarget', 'Fouls', 'Corners', 'YellowCards', 'RedCards',
        'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A' # Example betting odds
    ])

    print("\nLoading and combining main Premier League data...")
    combined_football_data = load_and_combine_data(DATA_DIR, CSV_FILES, columns_to_keep=main_df_relevant_columns)

    if combined_football_data.empty:
        print("Failed to load any main Premier League data. Exiting.")
    else:
        print(f"\nSuccessfully loaded {len(combined_football_data)} rows of historical Premier League data.")
        print("First 5 rows of combined main data:")
        try:
            from IPython.display import display
            display(combined_football_data.head())
        except ImportError:
            print(combined_football_data.head().to_string())

        # List all sample files
        sample_files = [f for f in os.listdir(SAMPLE_DATA_DIR) if f.startswith('sample_') and f.endswith('_table.csv')]
        sample_numbers = []
        for f in sample_files:
            try:
                num_part = f.replace('sample_', '').replace('_table.csv', '')
                sample_numbers.append(int(num_part))
            except ValueError:
                continue # Skip files that don't match the expected numeric pattern
        sample_numbers.sort()

        print("\n--- Matching Sample Files to Historical Data ---")
        results = []
        total_samples_processed = 0
        total_matches_found = 0
        total_no_conclusive_result = 0

        # Define columns to display for the matched historical data
        display_cols_for_match = [
            'Season', 'Date', 'HomeTeam', 'AwayTeam', 'FTR', 'FTHG', 'FTAG',
            'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
        ]

        for sample_num in sample_numbers:
            total_samples_processed += 1
            dynamic_target_stats, _ = populate_target_stats_from_csv(sample_num, SAMPLE_DATA_DIR)

            mapping_status = "No Conclusive Result"
            match_details_string = ""
            if dynamic_target_stats:
                matching_matches = find_matches_by_stats(combined_football_data, dynamic_target_stats)

                if not matching_matches.empty:
                    if len(matching_matches) == 1:
                        total_matches_found += 1
                        match_info = matching_matches.iloc[0]
                        mapping_status = (
                            f"{match_info['HomeTeam']} vs {match_info['AwayTeam']} on {match_info['Date']} "
                            f"(Season: {match_info['Season']})"
                        )
                        # Format the match details into a readable string
                        match_details = []
                        for col in display_cols_for_match:
                            if col in match_info: # Ensure column exists
                                match_details.append(f"{col}: {match_info[col]}")
                        match_details_string = "; ".join(match_details)
                    else:
                        # For multiple matches, just indicate count and first few
                        mapping_status = f"Multiple Matches Found ({len(matching_matches)}): "
                        first_matches_info = []
                        for i, match_info in matching_matches.head(3).iterrows():
                            first_matches_info.append(
                                f"{match_info['HomeTeam']} vs {match_info['AwayTeam']} on {match_info['Date']} (Season: {match_info['Season']})"
                            )
                        mapping_status += "; ".join(first_matches_info)
                        if len(matching_matches) > 3:
                            mapping_status += " and more..."
                        match_details_string = "N/A (Multiple Matches)" # Indicate no specific details for multiple
                else:
                    total_no_conclusive_result += 1
                    match_details_string = "N/A"
            else:
                total_no_conclusive_result += 1 # Count if sample file was not parsed correctly or not found
                match_details_string = "N/A (Sample parsing error or file not found)"

            results.append({
                'Sample #': sample_num,
                'Mapping': mapping_status,
                'Match Stats (Historical Data)': match_details_string
            })

        results_df = pd.DataFrame(results)

        print("\n--- Automated Match Mapping Results ---")
        try:
            from IPython.display import display
            display(results_df)
        except ImportError:
            # Use to_string() for console display to ensure full output
            print(results_df.to_string())

        print(f"\n--- Summary ---")
        print(f"Total samples processed: {total_samples_processed}")
        print(f"Total matches found (exactly one match per sample): {total_matches_found}")
        print(f"Total 'No Conclusive Result' samples: {total_no_conclusive_result}")

        # --- NEW CODE TO GENERATE 'final_df' ---
        print("\n--- Generating final_df with dataset.json mapping ---")
        id_mapping = {}
        try:
            with open(DATASET_JSON_PATH, 'r') as file:
                dataset = json.load(file)
            id_mapping = {entry['id']: entry for entry in dataset}
            print(f"Loaded {len(id_mapping)} entries from dataset.json for ID mapping.")
        except FileNotFoundError:
            print(f"Error: {DATASET_JSON_PATH} not found. Cannot perform ID mapping for final_df.")
        except json.JSONDecodeError:
            print(f"Error: Could not decode JSON from {DATASET_JSON_PATH}.")

        # Filter the results for rows with a value other than "No Conclusive Result"
        # Using the 'results' list generated earlier in this script
        filtered_results = [row for row in results if row['Mapping'] != "No Conclusive Result"]
        print(f"Filtered results (not 'No Conclusive Result'): {len(filtered_results)} rows.")

        # Get the first 275 rows
        selected_results = filtered_results[:275]
        print(f"Selected first 275 filtered results: {len(selected_results)} rows.")

        # Create a DataFrame with the selected rows and their corresponding IDs
        selected_data = []
        for result in selected_results:
            sample_id = result.get('Sample #') # Use .get() for safer access
            if sample_id is not None and sample_id in id_mapping:
                # Create a copy of the result dictionary to avoid modifying original
                # and then add the 'ID' from id_mapping
                row_data = result.copy()
                row_data['ID'] = id_mapping[sample_id]['id']
                selected_data.append(row_data)
            else:
                # Optional: print a message if a sample_id is not found in id_mapping
                if sample_id is not None:
                    print(f"Warning: Sample ID {sample_id} from results not found in dataset.json for final_df.")

        final_df = pd.DataFrame(selected_data)

        # Display the resulting DataFrame
        print("\n--- Head of final_df ---")
        try:
            from IPython.display import display
            display(final_df.head())
        except ImportError:
            print(final_df.head().to_string())

        print(f"\nFinal DataFrame 'final_df' has {len(final_df)} rows.")

Using main data directory: ./LiveSum_++/english-premier-league/
Using sample data directory: ./LiveSum_++/training_data/
CSV files to process (main data): ['2013-14.csv', '2014-15.csv', '2015-16.csv', '2016-17.csv', '2017-18.csv', '2018-19.csv', '2019-20.csv', '2020-21.csv', '2021-22.csv', '2022-23.csv', '2023-24.csv']

Loading and combining main Premier League data...

Successfully loaded 3840 rows of historical Premier League data.
First 5 rows of combined main data:


,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR,B365H,B365D,B365A,Season
0,E0,17/08/13,Arsenal,Aston Villa,1,3,A,1,1,D,A Taylor,16,9,4,4,15,18,4,3,4,5,1,0,1.44,4.75,8.0,2013-14
1,E0,17/08/13,Liverpool,Stoke,1,0,H,1,0,H,M Atkinson,26,10,11,4,11,11,12,6,1,1,0,0,1.40,5.00,9.5,2013-14
2,E0,17/08/13,Norwich,Everton,2,2,D,0,0,D,M Oliver,8,19,2,6,13,10,6,8,2,0,0,0,3.20,3.40,2.4,2013-14
3,E0,17/08/13,Sunderland,Fulham,0,1,A,0,0,D,N Swarbrick,20,5,3,1,14,14,6,1,0,3,0,0,2.30,3.40,3.4,2013-14
4,E0,17/08/13,Swansea,Man United,1,4,A,0,2,A,P Dowd,17,15,6,7,13,10,7,4,1,3,0,0,4.20,3.50,2.0,2013-14



--- Matching Sample Files to Historical Data ---

--- Automated Match Mapping Results ---


,Sample #,Mapping,Match Stats (Historical Data)
0,1,No Conclusive Result,N/A
1,2,Man United vs Tottenham on 01/01/14 (Season: 2...,Season: 2013-14; Date: 01/01/14; HomeTeam: Man...
2,3,West Brom vs Newcastle on 01/01/14 (Season: 20...,Season: 2013-14; Date: 01/01/14; HomeTeam: Wes...
3,4,Liverpool vs Hull on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Liv...
4,5,Stoke vs Everton on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Sto...
5,6,Swansea vs Man City on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Swa...
6,7,Fulham vs West Ham on 01/01/14 (Season: 2013-14),Season: 2013-14; Date: 01/01/14; HomeTeam: Ful...
7,8,No Conclusive Result,N/A
8,9,Southampton vs West Brom on 11/01/14 (Season: ...,Season: 2013-14; Date: 11/01/14; HomeTeam: Sou...
9,10,Hull vs Chelsea on 11/01/14 (Season: 2013-14),Season: 2013-14; Date: 11/01/14; HomeTeam: Hul...



--- Summary ---
Total samples processed: 3771
Total matches found (exactly one match per sample): 2718
Total 'No Conclusive Result' samples: 1053

--- Generating final_df with dataset.json mapping ---
Loaded 3771 entries from dataset.json for ID mapping.
Filtered results (not 'No Conclusive Result'): 2718 rows.
Selected first 275 filtered results: 275 rows.

--- Head of final_df ---


""



Final DataFrame 'final_df' has 0 rows.


In [2]:
import json

# Load the dataset.json file
with open('./LiveSum_++/dataset.json', 'r') as file:
    dataset = json.load(file)

# Create a dictionary mapping IDs to their corresponding data
id_mapping = {entry['id']: entry for entry in dataset}

# Filter the results for rows with a value other than "No Conclusive Result"
filtered_results = [row for row in results if row['Mapping'] != "No Conclusive Result"]

# Get the first 400 rows
selected_results = filtered_results[:400]

# Create a DataFrame with the selected rows and their corresponding IDs
selected_data = []
for result in selected_results:
    sample_id = result['Sample #']
    if sample_id in id_mapping:
        selected_data.append({
            'ID': id_mapping[sample_id]['id'],
            **result
        })

final_df = pd.DataFrame(selected_data)

# Display the resulting DataFrame
print(final_df.head())

Empty DataFrame
Columns: []
Index: []
